# 06 - Multi-Agent Clinical Reasoning & Medical RAG Evaluation
## MedAgent: Multimodal Clinical Decision Support System

This notebook demonstrates the end-to-end multi-agent clinical decision support pipeline:
- `SupervisorAgent`: Master workflow orchestrator
- `SignalAgent`: Rhythm morphology analysis
- `ImagingAgent`: Radiographic finding & Grad-CAM ROI extraction
- `EvidenceAgent`: Semantic RAG retrieval of clinical practice guidelines
- `RiskAgent`: Composite severity scoring
- `ReportAgent`: Standardized Clinical Decision Support report synthesis


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.agents.supervisor_agent import SupervisorAgent
from src.data.loaders.image_loader import generate_synthetic_xray_image


### 1. Initialize Supervisor Agent & Specialized Subagents


In [ ]:
supervisor = SupervisorAgent(load_checkpoints=True)


### 2. Simulate Acute Clinical Patient Case


In [ ]:
patient_profile = {
    "patient_id": "PT-9402",
    "age": 71,
    "sex": "Male",
    "sbp": 164,
    "dbp": 96,
    "heart_rate": 102.0,
    "spo2": 89.0,
    "has_hypertension": 1,
    "has_diabetes": 1,
    "medical_history": "Presenting with acute shortness of breath, bilateral lower extremity edema, and palpitations.",
}

# 260-sample synthetic PVC beat
t = np.linspace(0, 1, 260)
dummy_ecg = (-0.4 * np.exp(-((t - 0.35) ** 2) / 0.002)
             + 1.6 * np.exp(-((t - 0.45) ** 2) / 0.004)
             - 0.8 * np.exp(-((t - 0.58) ** 2) / 0.006)).astype(np.float32)

dummy_xray = generate_synthetic_xray_image(pathology="Pleural Effusion")


### 3. Run Multi-Agent Diagnostic Execution Graph


In [ ]:
result = supervisor.run_case(
    patient_info=patient_profile,
    ecg_input=dummy_ecg,
    xray_input=dummy_xray
)

print(f"Analysis Complete! Stratified Risk: {result['risk_level']}")


### 4. Display Finalized Evidence-Grounded Clinical Decision Report


In [ ]:
from IPython.display import display, Markdown

display(Markdown(result["markdown_report"]))
